# B2-020 — Language Transformers

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F4-matrices`, `book1:C7-pytorch`, `book1:C10-cnn-transfer`, `book1:C8-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/01-F1-scientific-python/lesson.ipynb), [book1:F4-matrices](../../../book1/units/04-F4-matrices/lesson.ipynb), [book1:C7-pytorch](../../../book1/units/13-C7-pytorch/lesson.ipynb), [book1:C10-cnn-transfer](../../../book1/units/17-C10-cnn-transfer/lesson.ipynb), [book1:C8-neural-training](../../../book1/units/15-C8-neural-training/lesson.ipynb), [B2-019-attention-transformers](../B2-019-attention-transformers/lesson.ipynb).

This double-length unit trains embeddings, runs causal and masked pretraining, fine-tunes a committed tiny encoder, and designs auditable NLP applications.


## Goals

By the end, you can distinguish fixed from learned embeddings, build a causal language Transformer, compare causal and masked objectives, load a trained encoder state safely, run freeze/unfreeze fine-tuning, and choose heads/losses/metrics for classify, tag, generate, and retrieve tasks.

## Session map

1. Train token embeddings.
2. Causal Transformer language model.
3. Pretraining objectives and deterministic state generation.
4. Fine-tune a language Transformer.
5. Language-task design and leakage audit.

## How to study

Run lessons in order, complete both checkpoints in every numbered section, then use Sets A/B/C for retrieval, implementation, and integration. The bridge is diagnostic: follow its qualified remediation links when a prerequisite is not fluent.

## Boundaries

All data are tiny, literal, synthetic, CPU-only, and seeded. No external corpus, hidden state, tokenizer library, model hub, pretrained download, or Book 1 content is used. B2-019 supplies attention and causal-mask mechanics; this unit applies them to language objectives.

<!-- api-card -->
## Code in this unit

A quick reference for the library calls in this unit's code. Each one is explained in a comment (or in the text beside the code) where it first appears.

**New in this unit:**

- `torch.get_num_threads` — the number of CPU threads torch uses
- `nn.GELU` — the GELU activation `x · Φ(x)` as a layer
- `.data_ptr` — the memory address where a tensor's data starts; the same address means shared memory
- `nn.Embedding` — a trainable lookup table: row `i` is the vector for token id `i`
- `nn.MultiheadAttention` — multi-head attention layer (width, number of heads; `batch_first=True`)
- `torch.triu` — keeps the entries on and above a diagonal (`diagonal=1`: strictly above)
- `torch.exp` — `e` raised to each entry
- `math.log` — the natural logarithm of a Python number
- `.log_softmax` — the log of the softmax along `dim`, computed stably
- `.gather` — picks, from each row, the entry at a given index along `dim`
- `torch.full_like` — a tensor shaped like the given one, every entry one value (e.g. `-100`)
- `.union` — a set holding every element of the given sets
- `importlib.util.spec_from_file_location` — a description of how to load a Python file as a module
- `importlib.util.module_from_spec` — a new, empty module object for that description
- `random.seed` — seeds Python's built-in random generator
- `F.cosine_similarity` — the cosine of the angle between vectors, along `dim`
- `torch.topk` — the `k` largest entries and their indices (`.indices`)
- `torch.optim.AdamW` — Adam with decoupled weight decay
- `torch.set_num_threads` — how many CPU threads torch may use
- `.eq` — `==`, entry by entry
- `F.gelu` — the GELU function `x · Φ(x)`, applied to each entry
- `.canonical_encoder_state_hash` — the unit's data-module hash of the saved encoder tensors
- `.float` — the tensor converted to float32 (True becomes 1.0)

**Also used here (met earlier):**

- `torch.optim.SGD` — plain gradient-descent optimizer (first explained in Book 1 C8)
- `.parameters` — a module's trainable tensors (first explained in Book 1 C7)
- `.zero_grad` — clear old gradients (first explained in Book 1 C8)
- `F.cross_entropy` — softmax cross-entropy on logits (first explained in B2-019)
- `.backward` — autograd fills each `.grad` (first explained in Book 1 C8)
- `.step` — one optimizer update (first explained in Book 1 C8)
- `.load_state_dict` — copy saved tensors into module (first explained in Book 1 C7)
- `torch.equal` — same shape and same entries (first explained in Book 1 C7)
- `.state_dict` — module tensors keyed by name (first explained in Book 1 C7)